#### ML ANOMALY DETECTION

##### 16.1 DOCUMENTAÇÃO DO ML ANOMALY DETECTION

###### Objetivo

O **ML Anomaly Detection** complementa o módulo 08 (Outlier Profile) com uma abordagem **multivariada e não supervisionada**, baseada em **K-Means distribuído (PySpark ML)**.

O método ajusta grupos de registros em um espaço numérico padronizado e calcula a distância de cada observação ao centro do grupo atribuído. Distâncias elevadas são sinalizadas como **anomalias candidatas**, não como erros confirmados.

A análise busca responder:

> Quais observações apresentam comportamento multivariado incomum em relação aos agrupamentos aprendidos, e qual é a frequência dessas sinalizações?

O módulo não altera o dataset original, não modifica o Data Quality Score e não depende de regras de negócio.

---

###### Exemplos

**Exemplo 1 — Anomalia multivariada**

Duas variáveis podem apresentar valores individualmente comuns, mas uma combinação incomum. A distância ao centro do agrupamento pode sinalizar esse comportamento para investigação.

**Exemplo 2 — Dados incompletos**

Linhas com valores nulos ou não finitos nas variáveis selecionadas não são pontuadas. A quantidade excluída é registrada como limitação da cobertura.

**Exemplo 3 — Execução desabilitada**

Com `ML_ENABLE = False`, o notebook valida sua configuração e prepara estruturas vazias, sem treinar modelos ou varrer o dataset para pontuação.

**Exemplo 4 — Limite de treinamento**

Com `ML_MAX_TRAIN_ROWS = 20000`, o treinamento usa no máximo esse número de registros elegíveis; a pontuação, quando habilitada, pode avaliar todos os registros elegíveis.

---

###### Principais análises

1. **Validação:** verifica entradas, parâmetros e atributos numéricos.
2. **Preparação:** converte atributos selecionados para `double` e filtra valores inválidos.
3. **Treinamento:** utiliza `VectorAssembler`, `StandardScaler` e `KMeans` nativos do Spark.
4. **Pontuação:** calcula distâncias euclidianas aos centros atribuídos sem Python UDF.
5. **Sinalização:** utiliza um quantil configurável da distância observada como limiar exploratório.
6. **Consolidação:** produz métricas, perfil por cluster e resumo executivo.
7. **Validação opcional:** executa testes sintéticos isolados quando solicitado.

---

###### Métodos de execução

**Execução opt-in:** o treinamento e a pontuação estão desabilitados por padrão (`ML_ENABLE = False`) para evitar processamento acidental.

**Biblioteca compartilhada:** o notebook 16 carrega o módulo 02, que deve carregar o módulo 00 atualizado com as dependências de ML. Não instala pacotes nem repete imports de bibliotecas compartilhadas.

**Amostragem de treinamento:** `sample()` e `limit()` reduzem o conjunto usado para ajuste, mas não tornam a seleção uma amostra estatisticamente perfeita; os resultados dependem da amostra e da semente.

**Pontuação distribuída:** o modelo prevê clusters no Spark e calcula distâncias por expressões SQL, sem trazer registros originais ao driver.

**Threshold:** o quantil empírico é estimado sobre as distâncias pontuadas. Não é probabilidade de erro, nem um limite de qualidade universal.

---

###### Tratamento de situações especiais

- **Menos de duas variáveis numéricas:** não há análise multivariada.
- **Colunas ausentes:** a execução é interrompida com mensagem explícita.
- **Valores nulos, NaN ou infinitos:** são excluídos da análise, sem imputação silenciosa.
- **Poucos registros elegíveis:** o treinamento não é executado.
- **Variáveis constantes:** podem reduzir a capacidade discriminatória do modelo.
- **Clusters vazios ou convergência limitada:** exigem interpretação e ajustes de parâmetros.
- **Amostragem limitada:** pode não representar eventos raros.
- **Execução completa em dataset grande:** pode demandar recursos consideráveis.
- **Dados sensíveis:** não são exportados para serviços externos; não há persistência por padrão.

---

###### Interpretação dos resultados

**`distancia_centro`:** distância euclidiana no espaço padronizado.

**`limiar_anomalia`:** quantil configurado das distâncias observadas.

**`sinalizacao_ml`:** indica distância estritamente superior ao limiar.

**`cluster`:** agrupamento atribuído pelo K-Means; não corresponde a categoria de negócio.

**Princípio fundamental:**

> Uma anomalia detectada por Machine Learning é uma evidência para investigação, não a confirmação de um erro de qualidade.

Os resultados não devem penalizar automaticamente o Data Quality Score.

---

###### Características do módulo

- **Distribuído:** utiliza algoritmos e expressões do PySpark.
- **Genérico:** não pressupõe domínio dos dados.
- **Não supervisionado:** dispensa rótulos de erro.
- **Controlável:** limites explícitos de colunas e treinamento.
- **Rastreável:** registra configurações, cobertura e limiar.
- **Não destrutivo:** não altera os dados de origem.
- **Sem dependências extras:** utiliza somente PySpark ML do ambiente.
- **Conservador:** permanece desabilitado até ativação explícita.

---

###### DataFrames de resultado

**`ml_anomaly_profile_df`**

Consolida quantidade de registros pontuados, sinalizações, percentual sinalizado e limiar de distância.

**`ml_anomaly_clusters_df`**

Apresenta contagens, distâncias médias e sinalizações por cluster.

**`ml_anomaly_summary_df`**

Disponibiliza o resumo executivo, os parâmetros e o estado da execução.

**`ml_anomaly_scores_df`**

Quando habilitado, contém apenas `cluster`, `distancia_centro` e `sinalizacao_ml`, sem os atributos originais.

---

###### Limitações

1. **Distância não é erro:** anomalias exigem validação contextual.
2. **K-Means pressupõe geometria euclidiana:** pode não capturar todos os tipos de anomalia.
3. **Padronização dependente da amostra:** distribuições diferentes alteram as distâncias.
4. **Treinamento limitado:** a amostra pode não conter comportamentos raros.
5. **Casos incompletos:** linhas com valores inválidos não são pontuadas.
6. **Custo de processamento:** treinar e pontuar milhões de registros pode ser demorado.
7. **Sem comparação temporal:** o módulo não detecta drift entre execuções.
8. **Sem persistência automática:** resultados são disponibilizados apenas na sessão atual.
9. **Sem aprovação automática:** o modelo não determina a qualidade global do dataset.

---

###### Resultado esperado

Ao final da execução, o módulo deverá permitir responder:

1. Quais atributos foram utilizados?
2. Quantos registros foram considerados no treinamento?
3. Quantos registros foram efetivamente pontuados?
4. Quantos registros foram excluídos por valores inválidos?
5. Qual limiar foi utilizado?
6. Qual percentual foi sinalizado?
7. Como as sinalizações se distribuem entre clusters?
8. Quais limitações afetam a interpretação?

**Observação:** a execução de testes e o treinamento devem permanecer opcionais. O módulo 16 complementa o Outlier Profile, sem substituí-lo.


In [0]:
%run "./02_DATA_PREPARATION"

In [0]:
# ============================================================
# 16.3 CONFIGURAÇÕES DO ML ANOMALY DETECTION
# ============================================================

# O QUE FAZ:
# Define limites explícitos para treinamento e pontuação.

# COMO FAZ:
# Mantém treinamento desabilitado até autorização do usuário.

# POR QUE É IMPORTANTE:
# Evita executar operações custosas involuntariamente.

# PERGUNTA RESPONDIDA:
# "Quais são os parâmetros e limites desta avaliação?"

ML_ENABLE = False
ML_RUN_TESTS = False
ML_SELECTED_COLUMNS = None          # None = primeiras numéricas elegíveis
ML_EXCLUDE_COLUMNS = []
ML_MAX_FEATURES = 5
ML_MAX_TRAIN_ROWS = 20000
ML_TRAIN_SAMPLE_FRACTION = 0.05      # (0, 1]; limita amostra antes de limit()
ML_K = 3
ML_SEED = 42
ML_MAX_ITER = 20
ML_ANOMALY_QUANTILE = 0.99
ML_QUANTILE_RELATIVE_ERROR = 0.01
ML_SCORE_ALL_VALID_ROWS = True       # False = pontua somente a amostra de treino
ML_CACHE_SCORING = False             # Cache opcional; desligado por padrão


In [0]:
# ============================================================
# 16.4 VALIDAÇÃO DE ENTRADAS E PARÂMETROS
# ============================================================
# O QUE FAZ:
# Confere contratos e seleciona as variáveis numéricas.

# COMO FAZ:
# Utiliza funções genéricas centralizadas no notebook 00.

# POR QUE É IMPORTANTE:
# Evita treinamentos inválidos ou seleção indevida de atributos.

# PERGUNTA RESPONDIDA:
# "O dataset possui atributos elegíveis e configurações válidas?"

required_ml = [
    "VectorAssembler", "StandardScaler", "KMeans", "vector_to_array",
    "dq_ml_validate_parameters", "dq_ml_numeric_columns", "dq_ml_quote_column",
]
missing_ml = [name for name in required_ml if name not in globals()]
if missing_ml:
    raise RuntimeError(
        "Atualize o notebook 00_FRAMEWORK_CONFIG com a extensão ML e confirme "
        "que o 01_DATA_INGESTION o carrega. Ausentes: " + ", ".join(missing_ml)
    )
if "df_prepared" not in globals() or not isinstance(df_prepared, DataFrame):
    raise ValueError("df_prepared não está disponível; valide o notebook 02.")

dq_ml_validate_parameters(
    ML_K, ML_SEED, ML_MAX_TRAIN_ROWS, ML_ANOMALY_QUANTILE, ML_MAX_FEATURES
)
if not (0 < ML_TRAIN_SAMPLE_FRACTION <= 1):
    raise ValueError("ML_TRAIN_SAMPLE_FRACTION deve estar em (0, 1].")
if not (0 <= ML_QUANTILE_RELATIVE_ERROR <= 1):
    raise ValueError("ML_QUANTILE_RELATIVE_ERROR deve estar em [0, 1].")
if ML_MAX_ITER < 1:
    raise ValueError("ML_MAX_ITER deve ser >= 1.")

available_numeric = dq_ml_numeric_columns(df_prepared, ML_EXCLUDE_COLUMNS)
selected = (
    available_numeric[:ML_MAX_FEATURES]
    if ML_SELECTED_COLUMNS is None
    else list(ML_SELECTED_COLUMNS)
)
if len(selected) != len(set(selected)):
    raise ValueError("ML_SELECTED_COLUMNS contém colunas repetidas.")
if len(selected) > ML_MAX_FEATURES:
    raise ValueError("Número de colunas superior a ML_MAX_FEATURES.")
unknown = [name for name in selected if name not in available_numeric]
if unknown:
    raise ValueError("Colunas ausentes, excluídas ou não numéricas: " + str(unknown))
if ML_ENABLE and len(selected) < 2:
    raise ValueError("Selecione pelo menos duas colunas numéricas para ML.")

print("ML habilitado:", ML_ENABLE, "| Atributos:", selected)


In [0]:
# ============================================================
# 16.5 PREPARAÇÃO DAS VARIÁVEIS
# ============================================================

# O QUE FAZ:
# Prepara apenas atributos numéricos selecionados.

# COMO FAZ:
# Converte para double e remove linhas com valores inválidos.

# POR QUE É IMPORTANTE:
# Garante vetores numéricos finitos sem imputação arbitrária.

# PERGUNTA RESPONDIDA:
# "Quais registros podem ser pontuados pelo modelo?"

ml_valid_df = None
ml_total_rows = None
ml_valid_rows = None
ml_excluded_rows = None
if ML_ENABLE:
    ml_total_rows = df_prepared.count()
    ml_cast = df_prepared.select(
        *[dq_ml_quote_column(c).cast("double").alias(f"_ml_f{i}") for i, c in enumerate(selected)]
    )
    valid_conditions = [
        F.col(f"_ml_f{i}").isNotNull()
        & ~F.isnan(F.col(f"_ml_f{i}"))
        & (F.abs(F.col(f"_ml_f{i}")) != float("inf"))
        for i in range(len(selected))
    ]
    predicate = valid_conditions[0]
    for condition in valid_conditions[1:]:
        predicate = predicate & condition
    ml_valid_df = ml_cast.filter(predicate)
    ml_valid_rows = ml_valid_df.count()
    ml_excluded_rows = ml_total_rows - ml_valid_rows
    if ml_valid_rows < ML_K:
        raise ValueError("Registros elegíveis insuficientes para ML_K.")
    print("Total:", ml_total_rows, "| Válidos:", ml_valid_rows,
          "| Excluídos:", ml_excluded_rows)
else:
    print("Preparação custosa não executada: ML_ENABLE=False.")

In [0]:
# ============================================================
# 16.6 AMOSTRAGEM E TREINAMENTO DISTRIBUÍDO
# ============================================================

# O QUE FAZ:
# Treina K-Means sobre variáveis numéricas padronizadas.

# COMO FAZ:
# Usa VectorAssembler, StandardScaler e KMeans do Spark ML.

# POR QUE É IMPORTANTE:
# Permite detecção exploratória multivariada sem coletar linhas ao driver.

# PERGUNTA RESPONDIDA:
# "Quais centros de comportamento foram aprendidos?"

ml_model = None
ml_scaler_model = None
ml_train_rows = 0
ml_train_scaled = None
ml_feature_cols = [f"_ml_f{i}" for i in range(len(selected))]
if ML_ENABLE:
    ml_train_base = (
        ml_valid_df.sample(False, ML_TRAIN_SAMPLE_FRACTION, ML_SEED)
        .limit(ML_MAX_TRAIN_ROWS)
    )
    ml_assembler = VectorAssembler(
        inputCols=ml_feature_cols, outputCol="_ml_features", handleInvalid="error"
    )
    ml_train_vectors = ml_assembler.transform(ml_train_base)
    ml_train_rows = ml_train_vectors.count()
    if ml_train_rows < ML_K:
        raise ValueError(
            "A amostra ficou menor que ML_K. Aumente ML_TRAIN_SAMPLE_FRACTION "
            "ou reduza ML_K."
        )
    ml_scaler_model = StandardScaler(
        inputCol="_ml_features", outputCol="_ml_scaled",
        withStd=True, withMean=False
    ).fit(ml_train_vectors)
    ml_train_scaled = ml_scaler_model.transform(ml_train_vectors)
    ml_model = KMeans(
        k=ML_K, seed=ML_SEED, maxIter=ML_MAX_ITER,
        featuresCol="_ml_scaled", predictionCol="cluster"
    ).fit(ml_train_scaled)
    print("Linhas de treinamento:", ml_train_rows,
          "| Clusters:", len(ml_model.clusterCenters()))
else:
    print("Treinamento não executado.")

In [0]:
# ============================================================
# 16.7 PONTUAÇÃO MULTIVARIADA
# ============================================================

# O QUE FAZ:
# Calcula a distância ao centro atribuído para cada linha pontuada.

# COMO FAZ:
# Utiliza funções SQL Spark e centros pequenos mantidos como literais.

# POR QUE É IMPORTANTE:
# Evita UDF Python e não expõe registros originais.

# PERGUNTA RESPONDIDA:
# "Qual é a distância multivariada de cada observação?"

ml_scored_raw_df = None
if ML_ENABLE:
    if ML_SCORE_ALL_VALID_ROWS:
        ml_vectors = ml_assembler.transform(ml_valid_df)
        ml_scaled = ml_scaler_model.transform(ml_vectors)
    else:
        ml_scaled = ml_train_scaled
    ml_pred = ml_model.transform(ml_scaled).select("cluster", "_ml_scaled")
    centers = [
        [float(value) for value in center]
        for center in ml_model.clusterCenters()
    ]
    centers_expr = F.array(
        *[F.array(*[F.lit(x) for x in center]) for center in centers]
    )
    ml_scored_raw_df = (
        ml_pred
        .withColumn("_ml_values", vector_to_array(F.col("_ml_scaled")))
        .withColumn("_ml_center", F.element_at(centers_expr, F.col("cluster") + F.lit(1)))
        .withColumn(
            "distancia_centro",
            F.sqrt(F.aggregate(
                F.zip_with(
                    F.col("_ml_values"), F.col("_ml_center"),
                    lambda x, y: (x - y) * (x - y)
                ),
                F.lit(0.0), lambda acc, value: acc + value
            ))
        )
        .select("cluster", "distancia_centro")
    )
    if ML_CACHE_SCORING:
        ml_scored_raw_df = ml_scored_raw_df.cache()
    print("Pontuação distribuída preparada.")
else:
    print("Pontuação não executada.")

In [0]:
# ============================================================
# 16.8 LIMIAR E SINALIZAÇÃO EXPLORATÓRIA
# ============================================================

# O QUE FAZ:
# Calcula limiar empírico e sinaliza distâncias elevadas.

# COMO FAZ:
# Usa approximateQuantile sobre as distâncias calculadas.

# POR QUE É IMPORTANTE:
# Produz um critério explícito, configurável e auditável.

# PERGUNTA RESPONDIDA:
# "Quais observações ultrapassam o quantil configurado?"

ml_anomaly_threshold = None
ml_anomaly_scores_df = None
if ML_ENABLE:
    quantiles = ml_scored_raw_df.approxQuantile(
        "distancia_centro",
        [ML_ANOMALY_QUANTILE],
        ML_QUANTILE_RELATIVE_ERROR
    )
    if not quantiles:
        raise ValueError("Não foi possível estimar o limiar de anomalia.")
    ml_anomaly_threshold = float(quantiles[0])
    ml_anomaly_scores_df = ml_scored_raw_df.withColumn(
        "sinalizacao_ml",
        F.col("distancia_centro") > F.lit(ml_anomaly_threshold)
    )
    print("Limiar estimado:", ml_anomaly_threshold)
else:
    ml_anomaly_scores_df = spark.createDataFrame(
        [], "cluster int, distancia_centro double, sinalizacao_ml boolean"
    )

In [0]:
# ============================================================
# 16.9 PERFIL CONSOLIDADO E CLUSTERS
# ============================================================

# O QUE FAZ:
# Consolida métricas de sinalização e distribuição por cluster.

# COMO FAZ:
# Agrega os registros pontuados diretamente no Spark.

# POR QUE É IMPORTANTE:
# Evita exposição de registros e permite análise executiva.

# PERGUNTA RESPONDIDA:
# "Qual a frequência e a distribuição das anomalias candidatas?"

profile_schema = (
    "registros_origem long, registros_elegiveis long, registros_excluidos long, "
    "registros_treinamento long, registros_pontuados long, sinalizacoes long, "
    "pct_sinalizado double, limiar_anomalia double, quantidade_atributos int"
)
cluster_schema = (
    "cluster int, registros long, sinalizacoes long, "
    "distancia_media double, distancia_maxima double"
)
if ML_ENABLE:
    ml_anomaly_clusters_df = (
        ml_anomaly_scores_df.groupBy("cluster")
        .agg(
            F.count("*").alias("registros"),
            F.sum(F.col("sinalizacao_ml").cast("long")).alias("sinalizacoes"),
            F.avg("distancia_centro").alias("distancia_media"),
            F.max("distancia_centro").alias("distancia_maxima")
        )
    )
    totals = ml_anomaly_clusters_df.agg(
        F.sum("registros").alias("n"),
        F.sum("sinalizacoes").alias("a")
    ).first()
    n, a = int(totals["n"] or 0), int(totals["a"] or 0)
    ml_anomaly_profile_df = spark.createDataFrame(
        [(
            int(ml_total_rows), int(ml_valid_rows), int(ml_excluded_rows),
            int(ml_train_rows), n, a,
            float(100.0 * a / n) if n else None,
            float(ml_anomaly_threshold), len(selected)
        )],
        schema=profile_schema
    )
else:
    ml_anomaly_clusters_df = spark.createDataFrame([], schema=cluster_schema)
    ml_anomaly_profile_df = spark.createDataFrame([], schema=profile_schema)

In [0]:
# ============================================================
# 16.10 RESUMO EXECUTIVO
# ============================================================

# O QUE FAZ:
# Registra configurações, estado e limitações da avaliação.

# COMO FAZ:
# Cria uma estrutura Spark compacta com os parâmetros efetivos.

# POR QUE É IMPORTANTE:
# Favorece interpretação e reprodutibilidade.

# PERGUNTA RESPONDIDA:
# "Em quais condições a avaliação foi executada?"

ml_status = "CONCLUIDO" if ML_ENABLE else "DESABILITADO"
ml_anomaly_summary_df = spark.createDataFrame(
    [(
        ml_status, ", ".join(selected), int(ML_K),
        int(ML_MAX_TRAIN_ROWS), float(ML_TRAIN_SAMPLE_FRACTION),
        float(ML_ANOMALY_QUANTILE), int(ML_SEED),
        bool(ML_SCORE_ALL_VALID_ROWS),
        "Distância K-Means: sinalização exploratória, não erro confirmado."
    )],
    schema=(
        "status string, atributos string, k int, limite_treino int, "
        "fracao_amostra double, quantil double, seed int, "
        "pontuacao_completa boolean, limitacao string"
    )
)
print("Status:", ml_status)

In [0]:
# ============================================================
# 16.11 EXIBIÇÃO CONTROLADA
# ============================================================

# O QUE FAZ:
# Exibe apenas resultados agregados e metadados.

# COMO FAZ:
# Apresenta DataFrames pequenos, sem registros de origem.

# POR QUE É IMPORTANTE:
# Facilita inspeção sem gerar coleta desnecessária de dados.

# PERGUNTA RESPONDIDA:
# "Quais resultados consolidados foram produzidos?"

display(ml_anomaly_summary_df)
if ML_ENABLE:
    display(ml_anomaly_profile_df)
    display(ml_anomaly_clusters_df.orderBy("cluster"))
else:
    print("Ative ML_ENABLE=True para gerar indicadores analíticos.")

In [0]:
# ============================================================
# 16.12 TESTES SINTÉTICOS OPCIONAIS
# ============================================================

# O QUE FAZ:
# Valida funções compartilhadas e contratos de saída.

# COMO FAZ:
# Usa parâmetros e DataFrame sintético independente do dataset.

# POR QUE É IMPORTANTE:
# Detecta regressões sem reexecutar os Profiles.

# PERGUNTA RESPONDIDA:
# "Os contratos essenciais permanecem consistentes?"

if ML_RUN_TESTS:
    dq_ml_validate_parameters(2, 42, 100, 0.95, 2)
    synthetic = spark.createDataFrame(
        [(1.0, 2.0, "a"), (2.0, 3.0, "b")],
        "x double, y double, label string"
    )
    assert dq_ml_numeric_columns(synthetic) == ["x", "y"]
    assert "status" in ml_anomaly_summary_df.columns
    assert "sinalizacao_ml" in ml_anomaly_scores_df.columns
    assert "limiar_anomalia" in ml_anomaly_profile_df.columns
    print("Testes sintéticos de contratos aprovados.")
else:
    print("Testes desabilitados.")

In [0]:
# ============================================================
# 16.13 CONTRATOS E SAÍDAS FINAIS
# ============================================================

# O QUE FAZ:
# Confere os DataFrames publicados pelo módulo.

# COMO FAZ:
# Valida presença de colunas e estado de execução.

# POR QUE É IMPORTANTE:
# Preserva a integração com os próximos módulos.

# PERGUNTA RESPONDIDA:
# "As saídas esperadas estão disponíveis?"

assert {"status", "atributos", "quantil"}.issubset(ml_anomaly_summary_df.columns)
assert {"registros_pontuados", "sinalizacoes"}.issubset(ml_anomaly_profile_df.columns)
assert {"cluster", "registros", "sinalizacoes"}.issubset(ml_anomaly_clusters_df.columns)
assert {"cluster", "distancia_centro", "sinalizacao_ml"}.issubset(ml_anomaly_scores_df.columns)
print("Contratos do módulo 16 validados. Estado:", ml_status)